<a href="https://colab.research.google.com/github/meiladrahmani556/marine-cbm-ml-dissertation/blob/main/Notebook_1_Click/Notebook05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
PROJECT_PATH = "/content/drive/MyDrive/✨CBM Data for Marine System Monitoring & Analysis✨"

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

print('Libraries imported successfully')

Libraries imported successfully


In [4]:
from google.colab import files

uploaded = files.upload()

Saving Conditional_Base_Monitoring in Marine_System.csv to Conditional_Base_Monitoring in Marine_System.csv


In [5]:
df = pd.read_csv('Conditional_Base_Monitoring in Marine_System.csv')

df.columns = df.columns.str.strip()
df = df.apply(pd.to_numeric, errors='coerce')
df = df.dropna()
df = df.drop_duplicates()

print('Shape:', df.shape)

Shape: (11936, 18)


In [6]:
target_column = 'GT Compressor decay state coefficient'

X = df.drop(columns=[target_column])
y = df[target_column]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print('Train:', X_train.shape[0], '| Test:', X_test.shape[0])

Train: 9548 | Test: 2388


In [7]:
baseline_rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)
baseline_rf.fit(X_train, y_train)

y_pred_baseline = baseline_rf.predict(X_test)
r2_baseline   = r2_score(y_test, y_pred_baseline)
rmse_baseline = np.sqrt(mean_squared_error(y_test, y_pred_baseline))

print('--- Baseline Random Forest ---')
print('R2:  ', round(r2_baseline,   6))
print('RMSE:', round(rmse_baseline, 6))

cv_scores = cross_val_score(
    baseline_rf, X_train, y_train,
    cv=5, scoring='r2', n_jobs=-1
)
print('\nCross-Validation R2 Scores:', cv_scores)
print('Mean CV R2:', round(cv_scores.mean(), 6))

--- Baseline Random Forest ---
R2:   0.998156
RMSE: 0.00064

Cross-Validation R2 Scores: [0.99742632 0.99659067 0.99710304 0.9975484  0.99751297]
Mean CV R2: 0.997236


In [9]:
param_grid = {
    'n_estimators': [200, 300],
    'max_depth':    [None, 20],
    'min_samples_split': [2],
    'min_samples_leaf':  [1]
}

grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid=param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print('Best Parameters:', grid_search.best_params_)
print('Best CV R2:     ', round(grid_search.best_score_, 6))

Fitting 3 folds for each of 4 candidates, totalling 12 fits
Best Parameters: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
Best CV R2:      0.996268


In [10]:
best_rf      = grid_search.best_estimator_
y_pred_tuned = best_rf.predict(X_test)

r2_tuned   = r2_score(y_test, y_pred_tuned)
rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
mae_tuned  = mean_absolute_error(y_test, y_pred_tuned)

print('--- Tuned Random Forest ---')
print('R2:  ', round(r2_tuned,   6))
print('MAE: ', round(mae_tuned,  6))
print('RMSE:', round(rmse_tuned, 6))

--- Tuned Random Forest ---
R2:   0.998192
MAE:  0.000399
RMSE: 0.000633


In [11]:
comparison = pd.DataFrame({
    'Model': ['Baseline RF', 'Tuned RF'],
    'R2':    [r2_baseline, r2_tuned],
    'RMSE':  [rmse_baseline, rmse_tuned]
})

print(comparison.to_string(index=False))

      Model       R2     RMSE
Baseline RF 0.998156 0.000640
   Tuned RF 0.998192 0.000633


In [12]:
train_r2 = r2_score(y_train, best_rf.predict(X_train))
test_r2  = r2_tuned

print('Training R2:', round(train_r2, 6))
print('Test R2:    ', round(test_r2,  6))

if train_r2 - test_r2 < 0.01:
    print('No significant overfitting detected')
else:
    print('Warning: possible overfitting')

Training R2: 0.999703
Test R2:     0.998192
No significant overfitting detected


In [13]:
import os

os.makedirs(f'{PROJECT_PATH}/models', exist_ok=True)

joblib.dump(best_rf, f'{PROJECT_PATH}/models/random_forest_tuned.pkl')
joblib.dump(scaler,  f'{PROJECT_PATH}/models/scaler.pkl')

print('Tuned model saved to Google Drive')

Tuned model saved to Google Drive
